# Intrinsic 3D effective-mass semiconductor model

**Part II skeleton — not submission-ready until every TODO is completed**

This notebook will implement and validate the three-dimensional chain

$$
\{E_C,E_V,\mathbf M_c,\mathbf M_v,g_v\}
\rightarrow g_{3D}(E)\rightarrow f(E,\mu,T)
\rightarrow n(\mu,T),p(\mu,T)\rightarrow \mu_i(T),n_i(T).
$$

It is separate from the 2D PhysKit validation in [`01_bloch2d_validation.ipynb`](01_bloch2d_validation.ipynb). Final concentrations here must be reported in cm$^{-3}$. Transport remains outside scope.


## Completion map

Complete sections in order. Plotting cells must consume calculated result objects and must not reproduce physics independently.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.constants import Boltzmann, electron_mass, electron_volt, hbar, Planck
from scipy.integrate import quad
from scipy.optimize import brentq

K_B_EV = Boltzmann / electron_volt
EV_TO_J = electron_volt
M3_TO_CM3 = 1e-6  # multiply m^-3 by this to obtain cm^-3

completion = pd.DataFrame([
    ("3D state model and units", "TODO"),
    ("3D conduction and valence DOS", "TODO"),
    ("stable Fermi-Dirac occupation", "TODO"),
    ("adaptive/converged carrier integrals", "TODO"),
    ("bracketed intrinsic neutrality", "TODO"),
    ("Boltzmann validation oracle", "TODO"),
    ("silicon control at 300 K", "TODO"),
    ("eight mandatory automated tests", "TODO"),
], columns=["deliverable", "status"])
completion


,deliverable,status
0,3D state model and units,TODO
1,3D conduction and valence DOS,TODO
2,stable Fermi-Dirac occupation,TODO
3,adaptive/converged carrier integrals,TODO
4,bracketed intrinsic neutrality,TODO
5,Boltzmann validation oracle,TODO
6,silicon control at 300 K,TODO
7,eight mandatory automated tests,TODO


## 1. Physical state model and energy reference

Use $E_V=0$ internally unless another reference is justified. Define each valley or band with its principal masses, edge energy, degeneracy, and whether degeneracy has already been absorbed into a quoted DOS mass.

**TODO:** instantiate `BandEdgeModel` objects from `src/semiconductor_model.py`, document SI conversions, and demonstrate that $E_C-E_V=E_g$.


In [2]:
state_model_schema = pd.DataFrame([
    ("conduction", "Ec", "eV", "TODO", "edge energy"),
    ("conduction", "m1,m2,m3", "m0", "TODO", "principal curvature masses"),
    ("conduction", "gv", "1", "TODO", "equivalent valleys"),
    ("valence", "Ev", "eV", "0.0", "internal energy reference"),
    ("valence", "m1,m2,m3", "m0", "TODO", "HH/LH or combined model"),
    ("valence", "gv", "1", "TODO", "band/valley multiplicity"),
], columns=["sector", "parameter", "unit", "value", "definition"])
state_model_schema


,sector,parameter,unit,value,definition
0,conduction,Ec,eV,TODO,edge energy
1,conduction,"m1,m2,m3",m0,TODO,principal curvature masses
2,conduction,gv,1,TODO,equivalent valleys
3,valence,Ev,eV,0.0,internal energy reference
4,valence,"m1,m2,m3",m0,TODO,HH/LH or combined model
5,valence,gv,1,TODO,band/valley multiplicity


## 2. Three-dimensional DOS

Implement and unit-test

$$
g_c(E)=\frac{g_sg_v}{4\pi^2}\left(\frac{2}{\hbar^2}\right)^{3/2}
\sqrt{\det\mathbf M_c}\sqrt{E-E_C}\,\Theta(E-E_C),
$$

and the analogous valence expression. Sum inequivalent valleys or valence bands explicitly.

**Required evidence:** units of states J$^{-1}$m$^{-3}$ (or eV$^{-1}$m$^{-3}$ after an explicit conversion), isotropic-mass test, and anisotropic determinant test.


In [3]:
dos_tasks = pd.DataFrame([
    ("dos_conduction(E, model)", "TODO", "sum all conduction components"),
    ("dos_valence(E, model)", "TODO", "sum all valence components"),
    ("DOS unit derivation", "TODO", "show concentration after integration"),
    ("isotropic oracle", "TODO", "tensor result equals scalar formula"),
    ("anisotropic oracle", "TODO", "tensor result equals determinant formula"),
], columns=["item", "status", "acceptance evidence"])
dos_tasks


,item,status,acceptance evidence
0,"dos_conduction(E, model)",TODO,sum all conduction components
1,"dos_valence(E, model)",TODO,sum all valence components
2,DOS unit derivation,TODO,show concentration after integration
3,isotropic oracle,TODO,tensor result equals scalar formula
4,anisotropic oracle,TODO,tensor result equals determinant formula


## 3. Occupation and carrier integrals

Implement a stable Fermi–Dirac function and calculate

$$n=\int_{E_C}^{\infty}g_c(E)f(E,\mu,T)dE,\qquad
p=\int_{-\infty}^{E_V}g_v(E)[1-f(E,\mu,T)]dE.$$

**TODO:** establish energy cutoffs with a convergence table rather than an undocumented constant. Record absolute/relative quadrature tolerances and support multiple band components.


In [4]:
integration_convergence_template = pd.DataFrame(columns=[
    "T (K)", "mu (eV)", "cutoff (kBT)", "quadrature tolerance",
    "n (cm^-3)", "p (cm^-3)", "relative change",
])
integration_convergence_template


,T (K),mu (eV),cutoff (kBT),quadrature tolerance,n (cm^-3),p (cm^-3),relative change


## 4. Intrinsic neutrality

Solve $n(\mu_i,T)-p(\mu_i,T)=0$ with a bracketing solver. Store the bracket, tolerances, iteration status, and final residual in a result object.


In [5]:
neutrality_output_schema = pd.DataFrame([
    ("temperature_K", "input"), ("mu_eV", "root"),
    ("n_cm3", "calculated"), ("p_cm3", "calculated"),
    ("bracket_eV", "solver evidence"), ("residual_cm3", "solver evidence"),
    ("relative_residual", "solver evidence"),
], columns=["field", "role"])
neutrality_output_schema


,field,role
0,temperature_K,input
1,mu_eV,root
2,n_cm3,calculated
3,p_cm3,calculated
4,bracket_eV,solver evidence
5,residual_cm3,solver evidence
6,relative_residual,solver evidence


## 5. Nondegenerate validation oracle

Implement independently:

$$N_C=2g_{v,e}\left(\frac{2\pi m_{c,d}k_BT}{h^2}\right)^{3/2},\quad
N_V=2g_{v,h}\left(\frac{2\pi m_{v,d}k_BT}{h^2}\right)^{3/2},$$

$$n_i=\sqrt{N_CN_V}e^{-E_g/(2k_BT)},\quad
\mu_i=\frac{E_C+E_V}{2}+\frac{k_BT}{2}\ln(N_V/N_C).$$

**TODO:** compare numerical Fermi–Dirac integrals with this oracle across temperature and identify breakdown of the approximation.


## 6. Required silicon control case

Use one internally consistent parameter source. The acceptance target at 300 K is approximately $E_g=1.12$ eV and $n_i=1.5	imes10^{10}$ cm$^{-3}$. Do not tune constants to force agreement; explain convention-dependent discrepancies.


In [6]:
silicon_control = pd.DataFrame([
    ("temperature", 300.0, "K", "assessment control"),
    ("target gap", 1.12, "eV", "approximate acceptance target"),
    ("target intrinsic concentration", 1.5e10, "cm^-3", "approximate acceptance target"),
    ("parameter source", np.nan, "citation", "TODO: one internally consistent source"),
], columns=["quantity", "value", "unit", "note"])
silicon_control


,quantity,value,unit,note
0,temperature,3.000000e+02,K,assessment control
1,target gap,1.120000e+00,eV,approximate acceptance target
2,target intrinsic concentration,1.500000e+10,cm^-3,approximate acceptance target
3,parameter source,NaN,citation,TODO: one internally consistent source


## 7. Required plots and comparison tables

- $g_c(E)$ and $g_v(E)$ with $f(E,\mu_i,T)$ at one temperature
- $\mu_i(T)$ relative to $E_V$, midgap, and $E_C$
- log-scale $n_i(T)$
- numerical versus nondegenerate table
- silicon control table
- validation table


In [7]:
required_outputs = pd.DataFrame([
    ("DOS plus occupation", "TODO"),
    ("mu_i versus temperature", "TODO"),
    ("ni versus temperature", "TODO"),
    ("numerical/Boltzmann table", "TODO"),
    ("silicon control table", "TODO"),
], columns=["output", "status"])
required_outputs


,output,status
0,DOS plus occupation,TODO
1,mu_i versus temperature,TODO
2,ni versus temperature,TODO
3,numerical/Boltzmann table,TODO
4,silicon control table,TODO


## 8. Mandatory automated tests


In [8]:
mandatory_tests = pd.DataFrame([
    ("n increases monotonically with mu", "TODO"),
    ("p decreases monotonically with mu", "TODO"),
    ("intrinsic neutrality residual below tolerance", "TODO"),
    ("rigid energy-shift invariance", "TODO"),
    ("isotropic tensor equals scalar-mass formula", "TODO"),
    ("diagonal anisotropic tensor equals determinant formula", "TODO"),
    ("Nc and Nv scale as T^(3/2)", "TODO"),
    ("energy cutoff and quadrature convergence", "TODO"),
], columns=["test", "status"])
mandatory_tests


,test,status
0,n increases monotonically with mu,TODO
1,p decreases monotonically with mu,TODO
2,intrinsic neutrality residual below tolerance,TODO
3,rigid energy-shift invariance,TODO
4,isotropic tensor equals scalar-mass formula,TODO
5,diagonal anisotropic tensor equals determinant...,TODO
6,Nc and Nv scale as T^(3/2),TODO
7,energy cutoff and quadrature convergence,TODO


## Part II conclusion — TODO

State the numerical integration range, neutrality tolerance, silicon discrepancy, and the temperature range over which the nondegenerate approximation is adequate. Do not include transport quantities.
